# 한↔영 양방향 번역기 학습 (Colab)

`C5_W4_A1_Transformer_Subclass_v1.ipynb` 에서 구현한 Transformer(`transformer.py`)로
한↔영 양방향 번역 모델을 학습하고, Android TV 앱용 TFLite 모델로 변환한다.

**준비**: 이 `training/` 폴더 전체를 Google Drive 에 올리고, 아래 `CODE_DIR` 를 그 경로로 바꾼다.
런타임 유형은 **GPU** (T4 이상)로 설정한다.

| 단계 | 내용 | 결과물 (WORK_DIR) |
|---|---|---|
| 1 | OPUS-100 en-ko 다운로드/정제, SentencePiece 학습 | `train.tsv`, `spm.model`, `tokenizer.vocab` |
| 2 | Transformer 학습 (세션이 끊겨도 다시 실행하면 이어서 학습) | `ckpt/` |
| 3 | TFLite 변환 + 검증 | `android_assets/` |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
CODE_DIR = '/content/drive/MyDrive/tv_translator/training'   # training/ 폴더를 올린 위치
WORK_DIR = '/content/drive/MyDrive/tv_translator/work'       # 데이터/체크포인트/결과 저장 위치
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
!pip install -q sentencepiece
!nvidia-smi -L

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*does not support masking.*")
import tensorflow as tf, keras
print(tf.__version__, keras.__version__, tf.config.list_physical_devices('GPU'))

# notebook 의 단위 테스트로 모델 코드 확인 (public_tests.py 가 CODE_DIR 에 있을 때)
if os.path.exists('public_tests.py'):
    from transformer import *
    from public_tests import *
    Transformer_test(Transformer, create_look_ahead_mask, create_padding_mask)

## 1단계: 데이터 준비
이미 만들어져 있으면 건너뛴다. AI Hub 등 추가 코퍼스가 있으면 `en<TAB>ko` TSV 로 만들어 `extra_tsv` 에 넣는다.

In [ ]:
import data_prep
if not os.path.exists(os.path.join(WORK_DIR, 'tokenizer.vocab')):
    data_prep.main(WORK_DIR, extra_tsv=[])
else:
    print('이미 준비됨:', os.listdir(WORK_DIR))

## 2단계: 학습
`config.py` 의 `TRAIN_STEPS`(기본 80,000)만큼 학습한다. 100 step 마다 속도(s/step)가 출력되므로
남은 시간을 보고 `train_steps` 를 조정하면 된다. 세션이 끊기면 이 셀을 다시 실행하면 마지막 checkpoint 부터 이어서 학습한다.

In [ ]:
import train
model, sp = train.main(WORK_DIR, train_steps=80000)

In [ ]:
# 학습된 Keras 모델로 몇 문장 번역해 보기
for text, to in [('How are you today?', 'ko'), ('I would like to book a hotel room for two nights.', 'ko'),
                 ('오늘 날씨가 정말 좋네요.', 'en'), ('이 근처에 맛있는 식당이 있나요?', 'en')]:
    print(f'[{to}] {text}\n   → {train.greedy_translate(model, sp, text, to)}')

## 3단계: TFLite 변환
`android_assets/` 에 `translator.tflite`, `tokenizer.vocab`, `model_config.json`, `tokenizer_testcases.tsv` 가 만들어진다.
TFLite 결과가 Keras 결과와 같은지(토큰 일치율)도 출력한다. (int8 양자화로 소수 문장은 달라질 수 있음)

In [ ]:
import export_tflite
out_dir = export_tflite.main(WORK_DIR, quantize=True, verify_n=50)
!ls -la {out_dir}

In [ ]:
# 앱과 같은 로직(문장 분리 + greedy)으로 TFLite 모델 확인
tl = export_tflite.TFLiteTranslator(open(os.path.join(out_dir, 'translator.tflite'), 'rb').read(),
                                    os.path.join(out_dir, 'tokenizer.vocab'))
print(tl.translate('Hello. Nice to meet you! Where is the subway station?', 'ko'))
print(tl.translate('안녕하세요. 만나서 반갑습니다! 지하철역이 어디에 있나요?', 'en'))

## 결과물 내려받기
압축을 풀어 다음 위치에 복사한 뒤 Android 프로젝트를 빌드한다.
- `translator.tflite`, `tokenizer.vocab`, `model_config.json` → `android/app/src/main/assets/`
- `tokenizer.vocab`, `tokenizer_testcases.tsv` → `android/app/src/test/resources/`

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/android_assets', 'zip', out_dir)
files.download(zip_path)